# Fase 1: Dataset e Feature engineerging per dati endogeni

# 1. Download dei dati contabili con yfinance

Selezione delle aziende da scarica da yfinance:
yfinance non supporta il download massivo di tutti i dati delle aziende quotate, per cui sono state scelte le top 30 aziende che compongono l'indice FTSE-MIB
di queste 30 aziende sono state selezionate solo le aziende Corporate, poiché le Finance hanno indici diversi
verrà creata un dizionario contenente {codice_ftse : nome azienda} al fine di far girare uno script per l'estrazione dei dati e il successivo inserimento in un dataframe pandas

In [1]:
# Creazione dizionario python con le 30 aziende scelte

ftse_mib_corporate = {
    "ENEL.MI": "Enel S.p.A.",
    "ENI.MI": "Eni S.p.A.",
    "RACE.MI": "Ferrari N.V.",
    "PRY.MI": "Prysmian S.p.A.",
    "STM.MI": "STMicroelectronics N.V.",
    "LDO.MI": "Leonardo S.p.A.",
    "PST.MI": "Poste Italiane S.p.A.",  # Business misto (Logistica/Servizi)
    "TRN.MI": "Terna S.p.A.",
    "SRG.MI": "Snam S.p.A.",
    "MONC.MI": "Moncler S.p.A.",
    "TEN.MI": "Tenaris S.A.",
    "STLAM.MI": "Stellantis N.V.",
    "LTMC.MI": "Lottomatica Group S.p.A.",
    "SPM.MI": "Saipem S.p.A.",
    "REC.MI": "Recordati S.p.A.",
    "IG.MI": "Italgas S.p.A.",
    "TPRO.MI": "Technoprobe S.p.A.",
    "A2A.MI": "A2A S.p.A.",
    "CPR.MI": "Davide Campari-Milano N.V.",
    "BZU.MI": "Buzzi S.p.A."
}

In [2]:
# importazione librerie e creazione dello script per l'estrazione dei dati

import time
import pandas as pd
import yfinance as yf

# 1. Script per l'estrazione dei dati
def data_extraction_ftse_quarterly(companies_dictionary):
    """
    Estrazione dei bilanci trimestrali (Balance Sheet, Income Statement e Cash Flow)
    delle aziende Corporate FTSE-MIB.
    """
    all_data_frames = []
    
    print(f"Inizio estrazione trimestrale per {len(companies_dictionary)} aziende corporate...")

    for ftse_code, company_name in companies_dictionary.items():
        print(f"🔄 Download: {company_name} ({ftse_code})")
        
        try:
            ticker = yf.Ticker(ftse_code)
            
            # Estrazione dei prospetti trimestrali
            bs = ticker.quarterly_balance_sheet
            is_stmt = ticker.quarterly_income_stmt
            cf = ticker.quarterly_cashflow

            # Controllo per eventuali aziende senza dati
            if bs is None or bs.empty or cf is None or cf.empty:
                print(f"Dati non disponibili per {ftse_code}")
                continue
                
            # Unione verticale dei prospetti per data
            union_statements = pd.concat([bs, is_stmt, cf], axis=0)
            
            # Rimozione delle eventuali righe duplicate nelle voci di bilancio
            union_statements = union_statements.loc[~union_statements.index.duplicated(keep='first')]
            
            # Traspozione dei dataset per avere le date come righe e i nomi degli indici come colonne
            df_transposed = union_statements.T
            
            # Formattazione dell'indice temporale
            df_transposed.index = pd.to_datetime(df_transposed.index)
            df_transposed.index.name = "Date"
            
            # Reset dell'indice per colonne pulite
            df_flat = df_transposed.reset_index()
            df_flat.insert(0, "FTSE_Code", ftse_code)
            df_flat.insert(1, "Company_Name", company_name)
            
            all_data_frames.append(df_flat)
            
        except Exception as e:
            print(f"Errore con {ftse_code}: {str(e)}")
            
        time.sleep(2.5)  # Pausa API impostata a 2.5 secondi
        
    if all_data_frames:
        final_df = pd.concat(all_data_frames, ignore_index=True)
        final_df = final_df.sort_values(by=["Company_Name", "Date"], ascending=[True, False])
        return final_df
    else:
        print("Nessun dato estratto")
        return pd.DataFrame()

In [3]:
# Estrazione per le aziende selezionate

df_raw = data_extraction_ftse_quarterly(ftse_mib_corporate)
print(f"\nShape finale del DataFrame estratto: {df_raw.shape}")
print(f"\nPrime 5 righe del Dtaframe estratto: {df_raw.head()}")

Inizio estrazione trimestrale per 20 aziende corporate...
🔄 Download: Enel S.p.A. (ENEL.MI)


C:\Users\chian\AppData\Local\Temp\ipykernel_31260\2342448276.py:34: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  union_statements = pd.concat([bs, is_stmt, cf], axis=0)


🔄 Download: Eni S.p.A. (ENI.MI)
🔄 Download: Ferrari N.V. (RACE.MI)
🔄 Download: Prysmian S.p.A. (PRY.MI)


C:\Users\chian\AppData\Local\Temp\ipykernel_31260\2342448276.py:34: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  union_statements = pd.concat([bs, is_stmt, cf], axis=0)


🔄 Download: STMicroelectronics N.V. (STM.MI)
Dati non disponibili per STM.MI
🔄 Download: Leonardo S.p.A. (LDO.MI)


C:\Users\chian\AppData\Local\Temp\ipykernel_31260\2342448276.py:34: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  union_statements = pd.concat([bs, is_stmt, cf], axis=0)


... (7 outputs left out)

C:\Users\chian\AppData\Local\Temp\ipykernel_31260\2342448276.py:34: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  union_statements = pd.concat([bs, is_stmt, cf], axis=0)


🔄 Download: Italgas S.p.A. (IG.MI)


C:\Users\chian\AppData\Local\Temp\ipykernel_31260\2342448276.py:34: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  union_statements = pd.concat([bs, is_stmt, cf], axis=0)


🔄 Download: Technoprobe S.p.A. (TPRO.MI)
Dati non disponibili per TPRO.MI
🔄 Download: A2A S.p.A. (A2A.MI)
Dati non disponibili per A2A.MI
🔄 Download: Davide Campari-Milano N.V. (CPR.MI)
Dati non disponibili per CPR.MI
🔄 Download: Buzzi S.p.A. (BZU.MI)
Dati non disponibili per BZU.MI

Shape finale del DataFrame estratto: (83, 274)

Prime 5 righe del Dtaframe estratto:   FTSE_Code Company_Name       Date  Treasury Shares Number  \
5   ENEL.MI  Enel S.p.A. 2026-06-30             243416292.0   
4   ENEL.MI  Enel S.p.A. 2026-03-31             243416292.0   
3   ENEL.MI  Enel S.p.A. 2025-12-31             133554875.0   
2   ENEL.MI  Enel S.p.A. 2025-09-30              91622961.0   
1   ENEL.MI  Enel S.p.A. 2025-06-30              12079670.0   

   Ordinary Shares Number  Share Issued      Net Debt    Total Debt  \
5            9.923264e+09  1.016668e+10  6.567400e+10  7.313100e+10   
4            9.923264e+09  1.016668e+10  6.587600e+10  7.036500e+10   
3            1.003313e+10  1.016668e+1

# 2. Analisi Esplorativa dei Dati (EDA) & Pulizia

Il dataframe di pandas contiene 80 righe x 274 colonne, ma non tutte servono per l'analisi.
Prima di procedere all'EDA è importante verificare quali colonne sono effettivamente necessarie e filtrare il dataframe solo con quelle necessarie: liquidità (cash), debiti (libailities), crediti (receivables) e immobilizzazioni (fixed assets)
Una volta creato il dataset verifico quali sono le principali statistiche e i valori nulli. 
In base a tali valori verrà scelto il metodo di riempimento dei valori nulli e verranno calcolate le statistiche finali.

In [4]:
# 1. Estrazione delle feature significative per l'analisi

# Verifica dei nomi delle colonne per consentire l'estrazione
print(list(df_raw.columns))

# Definizione delle colonne base necessarie per l'analisi
colonne_base = ["FTSE_Code", "Company_Name", "Date"]

# Definizione delle colonne necessarie sulla base di parole chiave per ciascuna categoria
cash_cols = [c for c in df_raw.columns if any(k in c.lower() for k in ["cash", "equivalents", "short term investments"])]
liability_cols = [c for c in df_raw.columns if any(k in c.lower() for k in ["debt", "liabilities", "payable", "borrowing", "lease"])]
receivable_cols = [c for c in df_raw.columns if any(k in c.lower() for k in ["receivable", "working capital"])]
fixed_assets_cols = [c for c in df_raw.columns if any(k in c.lower() for k in ["ppe", "property", "plant", "equipment", "intangible", "goodwill", "non current assets", "investment"])]

# Unione delle estrazioni
all = list(set(colonne_base + cash_cols + liability_cols + receivable_cols + fixed_assets_cols))

# Creazione del dtaframe filtrato
df_liquidity_analysis = df_raw[[c for c in all if c in df_raw.columns]].copy()

# Formattazione delle colonne con valori numerici per visualizzazione migliore
money_col = [c for c in df_liquidity_analysis.columns if c not in ['FTSE_Code', 'Company_Name', 'Date']]
df_liquidity_analysis_scaled = df_liquidity_analysis.copy()
# Dividiamo per 1 miliardo (1e9)
df_liquidity_analysis_scaled[money_col] = df_liquidity_analysis_scaled[money_col] / 1e9

print(f"Dimensioni del DataFrame filtrato: {df_liquidity_analysis.shape}")


['FTSE_Code', 'Company_Name', 'Date', 'Treasury Shares Number', 'Ordinary Shares Number', 'Share Issued', 'Net Debt', 'Total Debt', 'Tangible Book Value', 'Invested Capital', 'Working Capital', 'Net Tangible Assets', 'Capital Lease Obligations', 'Common Stock Equity', 'Total Capitalization', 'Total Equity Gross Minority Interest', 'Minority Interest', 'Stockholders Equity', 'Other Equity Interest', 'Treasury Stock', 'Retained Earnings', 'Additional Paid In Capital', 'Capital Stock', 'Common Stock', 'Total Liabilities Net Minority Interest', 'Total Non Current Liabilities Net Minority Interest', 'Other Non Current Liabilities', 'Derivative Product Liabilities', 'Non Current Pension And Other Postretirement Benefit Plans', 'Non Current Accrued Expenses', 'Tradeand Other Payables Non Current', 'Non Current Deferred Revenue', 'Non Current Deferred Taxes Liabilities', 'Long Term Debt And Capital Lease Obligation', 'Long Term Capital Lease Obligation', 'Long Term Debt', 'Long Term Provisions

In [5]:
# 2. Calcolo statistiche descrittive e verifica valori mancanti

# Statistiche descrittive
print("\nStatistiche descrittive di base (in € mld):")
cols_to_show = ["mean", "std", "min", "max"]
print(df_liquidity_analysis_scaled.describe().T[cols_to_show].round(2)) # trasposizione per migliorare visibilità e limite a 2 cifre decimali

# Verifica valori mancanti ordinato per colonne che hanno più valori mancanti
missing_values = df_liquidity_analysis_scaled.isnull().sum().sort_values(ascending=False)
print(f"\nNnumero di valori nulli per ciascuna colonna:{missing_values}")



Statistiche descrittive di base (in € mld):
                                                                      mean  \
Date                                            2025-12-06 21:06:30.361445   
Working Capital                                                  -3.022996   
Current Debt And Capital Lease Obligation                         4.459812   
Purchase Of Intangibles                                          -0.073666   
Long Term Capital Lease Obligation                                1.022283   
...                                                                    ...   
Non Current Deferred Liabilities                                  2.330825   
Net PPE Purchase And Sale                                        -0.663698   
Sale Of Intangibles                                               0.000763   
Cash Flow From Continuing Investing Activities                   -1.305995   
Long Term Debt Issuance                                           1.223214   

                  

Dall'analisi emerge che molte colonne hanno valori mancanti, alcune addirittura 78 su 80 righe. 
I valori mancanti verranno pertano gestiti nel seguente modo:
 - eliminazione delle colonne con più del 50% dei valori mancanti;
 - stima dei dati mancanti delle colonne residue con KNNImputer

In [14]:
# 3. Gestione valori mancanti

# Rimozione delle colonne con più del 50% dei valori mancanti
nan_treshold = len(df_liquidity_analysis_scaled) * 0.5
df_liquidiy_cleaned = df_liquidity_analysis_scaled.dropna(thresh=nan_treshold, axis=1)

print(f"Colonne prima della pulizia: {df_liquidity_analysis_scaled.shape[1]}")
print(f"Colonne post pulizia (soglia del 50%): {df_liquidiy_cleaned.shape[1]}")

# Stima dei valori mancanti per le colonne residue con KNNImputer

from sklearn.impute import KNNImputer

# creazione di colonne di metadati e numeriche

metadata_cols = [c for c in ["FTSE_Code", "Company_Name", "Date"] if c in df_liquidiy_cleaned.columns]
numeric_cols = [c for c in df_liquidiy_cleaned.columns if c not in metadata_cols]

# Inizializzazione del KNN Inputer (5 bilanci con caratteristiche più simili)
imputer = KNNImputer(n_neighbors=5)

df_imputed_numeric = pd.DataFrame(
    imputer.fit_transform(df_liquidiy_cleaned[numeric_cols]),
    columns=numeric_cols,
    index=df_liquidiy_cleaned.index
)

# Ricostruzione del dataframe finale
df_final_clean = pd.concat([df_liquidiy_cleaned[metadata_cols], df_imputed_numeric], axis=1)

# Verifica finale
print(f"Dimensioni dataset finale: {df_final_clean.shape}")
print(f"Valori nulli residui totali: {df_final_clean.isnull().sum().sum()}")
# Statistiche descrittive post pulizia
print("\nStatistiche descrittive post pulizia (in € mld):")
print(df_final_clean.describe().T[cols_to_show].round(2)) # trasposizione per migliorare visibilità e limite a 2 cifre decimali


Colonne prima della pulizia: 110
Colonne post pulizia (soglia del 50%): 52
Dimensioni dataset finale: (83, 52)
Valori nulli residui totali: 0

Statistiche descrittive post pulizia (in € mld):
                                                                          mean  \
Date                                                2025-12-06 21:06:30.361445   
Working Capital                                                      -2.598779   
Current Debt And Capital Lease Obligation                              4.55995   
Long Term Capital Lease Obligation                                    0.961469   
Total Debt                                                           18.746445   
Net Issuance Payments Of Debt                                          0.41852   
Gross PPE                                                            34.315815   
Taxes Receivable                                                       0.97697   
Long Term Equity Investment                                           

# 3. Feature Engineering
I dati di bilancio puliti rappresentano un buon punto di partenza, ma è utile estrarre da essi informazioni più qualitative attraverso la Feature Engineering.
I dati numerici vengono trasformati in rapporti così da diventare più significativi e allo stesso tempo risolvere i problemi legati a grandezze diverse attraverso la normalizzazione delle variabili. 
La Feature Engineering renderà anche pià facile l'introduzione di rumore stocastico e la stima di trend futuri  

In [11]:
# Verifica dei nomi delle colonne per capire quali informazioni estrarre
print(list(df_final_clean.columns))
df_features = df_final_clean.copy()

['FTSE_Code', 'Company_Name', 'Date', 'Working Capital', 'Current Debt And Capital Lease Obligation', 'Long Term Capital Lease Obligation', 'Total Debt', 'Net Issuance Payments Of Debt', 'Gross PPE', 'Taxes Receivable', 'Long Term Equity Investment', 'Goodwill And Other Intangible Assets', 'Other Receivables', 'Accounts Receivable', 'Non Current Deferred Taxes Liabilities', 'Accounts Payable', 'Financing Cash Flow', 'Payables', 'Net Long Term Debt Issuance', 'Current Liabilities', 'Net Debt', 'Investmentin Financial Assets', 'End Cash Position', 'Other Non Cash Items', 'Other Current Liabilities', 'Goodwill', 'Net PPE', 'Other Payable', 'Long Term Debt', 'Cash And Cash Equivalents', 'Total Tax Payable', 'Beginning Cash Position', 'Total Non Current Assets', 'Purchase Of PPE', 'Derivative Product Liabilities', 'Current Capital Lease Obligation', 'Capital Lease Obligations', 'Change In Working Capital', 'Investing Cash Flow', 'Total Unusual Items Excluding Goodwill', 'Other Intangible As

In [13]:
# Feature Engineering

# 1. Indicatori di liquidità statica breve termine

# Current Ratio = (Working Capital + Current Liabilities) /  Current Liabilities
df_features["current_ratio"] = (df_features["Working Capital"] + df_features["Current Liabilities"]) / df_features["Current Liabilities"]

# Quick Ratio = (Cash Cash Equivalents And Short Term Investments + Accounts Receivable) / Current Liabilities
df_features["quick_ratio"] = (df_features["Cash Cash Equivalents And Short Term Investments"] + df_features["Accounts Receivable"]) / df_features["Current Liabilities"] 

# Cash Ratio = Cash Cash Equivalents And Short Term Investments / Current Liabilities
df_features["cash_ratio"] = df_features["Cash Cash Equivalents And Short Term Investments"] / df_features["Current Liabilities"]

# Net Working Capital Ratio = Working Capital / (Total Non Current Assets + Working Capital + Current Liabilities)
df_features["NWCR_ratio"] = df_features["Working Capital"] / (df_features["Total Non Current Assets"] + df_features["Working Capital"] + df_features["Current Liabilities"])

# 2. Indicatori di liquidità statica dinamica

# Operating Cash Flow Ratio = Operating Cash Flow / Current Liabilities
df_features["OCF_ratio"] = df_features["Operating Cash Flow"] / df_features["Current Liabilities"]

# Free Cash Flow to Total Debt = Free Cash Flow / Total Debt
df_features["FCFTD_ratio"] = df_features["Free Cash Flow"] / df_features["Total Debt"]

# Cash Flow to Total Debt = Operating Cash Flow / Total Debt
df_features["CFTD_ratio"] = df_features["Operating Cash Flow"] / df_features["Total Debt"]

# 3. Solvibilità e leva finanziaria

# Debt to Liabilities Ratio = Total Debt / Total Liabilities Net Minority Interest
df_features["Debt_to_liabilities_ratio"] = df_features["Total Debt"] / df_features["Total Liabilities Net Minority Interest"]

# Net Debt to Operating Cash Flow = Net Debt / Operating Cash Flow
df_features["Net_debt_to_OFC_ratio"] = df_features["Net Debt"] / df_features["Operating Cash Flow"]

# 4. Lag Features: raggruppamenti dei valori del trimestre precedente per azienda
df_features["Cash_Lag_1"] = df_features.groupby("Company_Name")["Cash And Cash Equivalents"].shift(1)
df_features["FCF_Lag_1"] = df_features.groupby("Company_Name")["Free Cash Flow"].shift(1)

# 4. Tasso di crescita trimestrale del Cash Flow Operativo
df_features["OCF_Growth_QoQ"] = df_features.groupby("Company_Name")["Operating Cash Flow"].pct_change()


# 4. Data Augmentation
Inserimento di rumore stocastico nel dataset per simulare uno scenario di stress

In [20]:
import numpy as np

df_aug = df_features.copy()

# selezione delle colonne numeriche
numeric_cols_aug = [c for c in df_aug.columns if c not in metadata_cols]

# Data aumentation

np.random.seed(42)

noise_level = 0.02  #intensità del rumore bassa per evitare valori impossibili

# copia dei dati numerici
df_numeric_original = df_aug[numeric_cols_aug].copy()

# conversione in array numpy
data_values = df_numeric_original.values
std_values = df_numeric_original.std().values

# generazione del rumore gaussiano
noise = np.random.normal(0, 1, data_values.shape) * (std_values * noise_level)

# ricostruzione del DataFrame con i dati con rumori
df_numeric_noisy = pd.DataFrame(
    data_values + noise, 
    columns=numeric_cols_aug, 
    index=df_numeric_original.index
)

# unione dei dataframe
df_noisy_complete = pd.concat([df_aug[metadata_cols], df_numeric_noisy], axis=1)
df_final_augmented = pd.concat([df_aug, df_noisy_complete], ignore_index=True)

# pulizia di eventuali valori mancanti generati con la pulizia
df_final_augmented[numeric_cols_aug] = df_final_augmented[numeric_cols_aug].replace([np.inf, -np.inf], np.nan)
df_final_augmented[numeric_cols_aug] = df_final_augmented[numeric_cols_aug].fillna(0)

print(f"Dimensioni del DataFrame prima dell'augmentation: {df_aug.shape}")
print(f"Dimensioni del DataFrame finale aumentato: {df_final_augmented.shape}")

Dimensioni del DataFrame prima dell'augmentation: (83, 64)
Dimensioni del DataFrame finale aumentato: (166, 64)


# 5. Esportazione Dataset finale
Esportazione del dataset in .csv

In [21]:
df_final_augmented.to_csv("Dataset_analysis.csv",index=False)